In [34]:
import numpy as np
import pandas as pd

from find_tracts_in_range import find_tracts_in_range
from tract_group_delve_coverage import get_skymap, load_delve_csv, summarize_tract_group

In [27]:
step = 30

ra_min, ra_max = np.arange(270, 360 + 270 + 1, step=step) % 360, np.arange(270 + step, 360 + 270 + 1 + step, step=step) % 360
dec_min, dec_max = np.arange(30 - step, -61 - step, step=-step), np.arange(30, -61, step=-step)

ras, decs = np.array([ra_min, ra_max]).T, np.array([dec_min, dec_max]).T

patches = [
    dict(patch_id=i, ra_min=float(r[0]), ra_max=float(r[1]), dec_min=float(d[0]), dec_max=float(d[1]))
    for i, (r, d) in enumerate((r, d) for r in ras for d in decs)
]
print(f"{len(patches)} patches, {step} deg x {step} deg")

52 patches, 30 deg x 30 deg


In [28]:
skymap = get_skymap(skymap_name="lsst_cells_v2", repo="main")

In [29]:
patch_dfs = []
for patch in patches:
    df = find_tracts_in_range(
        patch["ra_min"], patch["ra_max"], patch["dec_min"], patch["dec_max"], skymap,
    )
    df.insert(0, "patch_id", patch["patch_id"])
    patch_dfs.append(df)
    print(f"patch {patch['patch_id']:2d}: RA [{patch['ra_min']:.1f}, {patch['ra_max']:.1f}] "
          f"Dec [{patch['dec_min']:.1f}, {patch['dec_max']:.1f}] -> {len(df)} tract(s)")

patch_tracts = pd.concat(patch_dfs, ignore_index=True)
print(f"\n{len(patch_tracts)} total (patch, tract) rows across {len(patches)} patches, "
      f"{patch_tracts['tract_id'].nunique()} distinct tracts")

patch  0: RA [270.0, 300.0] Dec [0.0, 30.0] -> 455 tract(s)
patch  1: RA [270.0, 300.0] Dec [-30.0, 0.0] -> 455 tract(s)
patch  2: RA [270.0, 300.0] Dec [-60.0, -30.0] -> 323 tract(s)
patch  3: RA [270.0, 300.0] Dec [-90.0, -60.0] -> 134 tract(s)
patch  4: RA [300.0, 330.0] Dec [0.0, 30.0] -> 450 tract(s)
patch  5: RA [300.0, 330.0] Dec [-30.0, 0.0] -> 450 tract(s)
patch  6: RA [300.0, 330.0] Dec [-60.0, -30.0] -> 323 tract(s)
patch  7: RA [300.0, 330.0] Dec [-90.0, -60.0] -> 141 tract(s)
patch  8: RA [330.0, 0.0] Dec [0.0, 30.0] -> 448 tract(s)
patch  9: RA [330.0, 0.0] Dec [-30.0, 0.0] -> 448 tract(s)
patch 10: RA [330.0, 0.0] Dec [-60.0, -30.0] -> 319 tract(s)
patch 11: RA [330.0, 0.0] Dec [-90.0, -60.0] -> 133 tract(s)
patch 12: RA [0.0, 30.0] Dec [0.0, 30.0] -> 448 tract(s)
patch 13: RA [0.0, 30.0] Dec [-30.0, 0.0] -> 448 tract(s)
patch 14: RA [0.0, 30.0] Dec [-60.0, -30.0] -> 319 tract(s)
patch 15: RA [0.0, 30.0] Dec [-90.0, -60.0] -> 133 tract(s)
patch 16: RA [30.0, 60.0] Dec [0

In [35]:
out_path = "patch_tracts.csv"
patch_tracts.to_csv(out_path, index=False)
print(f"wrote {out_path}")

wrote patch_tracts.csv


In [36]:
patch_tracts

,patch_id,tract_id,ra_lo,ra_hi,dec_lo,dec_hi,tract_area_deg2,overlap_area_deg2,frac_of_tract,frac_of_query
0,0,9408,268.888889,270.370370,-1.487603,0.000000,2.203609,0.000000e+00,0.000000e+00,0.000000e+00
1,0,9409,270.370370,271.851852,-1.487603,0.000000,2.203609,0.000000e+00,0.000000e+00,0.000000e+00
2,0,9410,271.851852,273.333333,-1.487603,0.000000,2.203609,0.000000e+00,0.000000e+00,0.000000e+00
3,0,9411,273.333333,274.814815,-1.487603,0.000000,2.203609,0.000000e+00,0.000000e+00,0.000000e+00
4,0,9412,274.814815,276.296296,-1.487603,0.000000,2.203609,0.000000e+00,0.000000e+00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...
17692,51,1335,288.292683,291.219512,-60.991736,-59.504132,2.160583,1.429493e+00,6.616238e-01,6.207485e-03
17693,51,1336,291.219512,294.146341,-60.991736,-59.504132,2.160583,1.429493e+00,6.616238e-01,6.207485e-03
17694,51,1337,294.146341,297.073171,-60.991736,-59.504132,2.160583,1.429493e+00,6.616238e-01,6.207485e-03
17695,51,1338,297.073171,300.000000,-60.991736,-59.504132,2.160583,1.429493e+00,6.616238e-01,6.207485e-03


In [37]:
DELVE_CSV = "../data/delve_dr3qr_combo_v10.csv"
delve_df = load_delve_csv(DELVE_CSV)
print(f"loaded {len(delve_df)} DELVE exposures from {DELVE_CSV}")

loaded 210989 DELVE exposures from ../data/delve_dr3qr_combo_v10.csv


In [ ]:
patch_summaries = []
for patch_id, group_df in patch_tracts.groupby("patch_id"):
    summary = summarize_tract_group(group_df, delve_df, skymap=skymap)
    summary.insert(0, "patch_id", patch_id)
    group = summary.attrs["group"]
    completeness = ", ".join(f"{row.band}={row.completeness_frac:.2f}" for row in summary.itertuples())
    print(f"patch {patch_id:2d}: {len(group['tract_ids'])} tract(s), {group['area_deg2']:.2f} deg^2 -> {completeness}")
    patch_summaries.append(summary)

patch_coverage = pd.concat(patch_summaries, ignore_index=True)
print(f"\n{len(patch_coverage)} rows across {patch_coverage['patch_id'].nunique()} patches")

patch  0: 427 tract(s), 939.31 deg^2 -> g=0.16, r=0.15, i=0.17, z=0.21
patch  1: 427 tract(s), 939.31 deg^2 -> g=0.49, r=0.26, i=0.49, z=0.35
patch  2: 316 tract(s), 688.53 deg^2 -> g=0.94, r=0.88, i=0.93, z=0.92
patch  3: 128 tract(s), 268.30 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch  4: 422 tract(s), 928.28 deg^2 -> g=0.56, r=0.56, i=0.62, z=0.55
patch  5: 422 tract(s), 928.28 deg^2 -> g=1.00, r=0.99, i=1.00, z=0.99
patch  6: 316 tract(s), 688.56 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch  7: 137 tract(s), 285.77 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch  8: 427 tract(s), 939.27 deg^2 -> g=0.99, r=0.99, i=0.90, z=0.99
patch  9: 427 tract(s), 939.27 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch 10: 319 tract(s), 695.06 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch 11: 131 tract(s), 273.66 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch 12: 425 tract(s), 934.87 deg^2 -> g=1.00, r=1.00, i=0.91, z=0.99
patch 13: 425 tract(s), 934.87 deg^2 -> g=1.00, r=1.00, i=1.00, z=1.00
patch 

In [ ]:
coverage_out_path = "patch_delve_coverage.csv"
patch_coverage.to_csv(coverage_out_path, index=False)
print(f"wrote {coverage_out_path}")